# 03 — Eksperimen CNN (Skenario 11)

MobileNetV2 two-stage fine-tuning, Grad-CAM, McNemar vs S6.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

from src.evaluate import (
    append_significance_test,
    build_gradcam_model,
    compute_metrics,
    make_gradcam_heatmap,
    mcnemar_test,
    plot_confusion_matrix,
    plot_gradcam,
    save_scenario_metrics,
)
from src.models import (
    build_mobilenetv2,
    compile_mobilenet,
    get_mobilenet_callbacks,
    unfreeze_last_layers,
)
from src.pipeline import image_to_cnn_input, process_image
from src.utils import build_dataset_index, get_project_paths, make_splits, read_best_enhancement, set_seed

set_seed(42)
paths = get_project_paths()
# Split di-regenerate dari dataset (deterministik) - identik dengan notebook 01/02
# RAW_DATA_DIR sudah di-set setup cell (auto-detect path Kaggle)
train_df, val_df, test_df = make_splits(build_dataset_index(RAW_DATA_DIR))
enhancement = read_best_enhancement(paths["metrics"])
print(f"Menggunakan enhancement E*: {enhancement}")


In [ ]:
# Cache hasil preprocessing (SSR + segmentasi) ke disk supaya tiap citra
# hanya diproses SEKALI, bukan diulang setiap epoch. Pakai /kaggle/temp
# (lega, ephemeral). Shuffle dipindah ke tf.data agar caching tetap benar.
CACHE_DIR = Path('/kaggle/temp/tfcache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def make_dataset(df, batch_size=32, shuffle=False, do_segment=True,
                 enhancement_method=None, cache_name=None):
    if enhancement_method is None:
        enhancement_method = enhancement
    def generator():
        label_map = {"fresh": 0, "rotten": 1}
        for _, row in df.iterrows():
            out = process_image(path=row["filepath"], enhancement=enhancement_method, do_segment=do_segment)
            if out["img"] is None:
                continue
            x = image_to_cnn_input(out["img"])[0]
            y = np.zeros(2, dtype=np.float32)
            y[label_map[row['label']]] = 1.0
            yield x, y

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(2,), dtype=tf.float32),
        )
    )
    if cache_name is not None:
        dataset = dataset.cache(str(CACHE_DIR / cache_name))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=2048, seed=42, reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Skenario 11: Dengan segmentasi + E*
train_ds_s11 = make_dataset(train_df, shuffle=True, do_segment=True, cache_name='train_s11')
val_ds_s11 = make_dataset(val_df, do_segment=True, cache_name='val_s11')
test_ds_s11 = make_dataset(test_df, do_segment=True, cache_name='test_s11')

# Skenario 12: Tanpa segmentasi (Citra Asli) + Tanpa Penajaman (none)
train_ds_s12 = make_dataset(train_df, shuffle=True, do_segment=False, enhancement_method='none', cache_name='train_s12')
val_ds_s12 = make_dataset(val_df, do_segment=False, enhancement_method='none', cache_name='val_s12')
test_ds_s12 = make_dataset(test_df, do_segment=False, enhancement_method='none', cache_name='test_s12')


In [ ]:
y_train_labels = train_df["label"].map({"fresh": 0, "rotten": 1}).values
classes = np.unique(y_train_labels)
weights = compute_class_weight("balanced", classes=classes, y=y_train_labels)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
class_weight


## Skenario 11: Pelatihan Model CNN Segmented + E*

### Stage 1 — Base frozen (20 epoch)

In [ ]:
model_s11 = build_mobilenetv2(num_classes=2)
model_s11 = compile_mobilenet(model_s11, learning_rate=1e-4)
cb_s11 = get_mobilenet_callbacks(str(paths["models"] / "mobilenetv2_s11_stage1.h5"))

history1_s11 = model_s11.fit(
    train_ds_s11, validation_data=val_ds_s11, epochs=20,
    class_weight=class_weight, callbacks=cb_s11,
)


### Stage 2 — Fine-tune 20 lapisan terakhir (50 epoch)

In [ ]:
model_s11 = unfreeze_last_layers(model_s11, n=20)
model_s11 = compile_mobilenet(model_s11, learning_rate=1e-5)
cb2_s11 = get_mobilenet_callbacks(str(paths["models"] / "mobilenetv2_s11_stage2.h5"))

history2_s11 = model_s11.fit(
    train_ds_s11, validation_data=val_ds_s11, epochs=50,
    class_weight=class_weight, callbacks=cb2_s11,
)


### Evaluasi Skenario 11

In [ ]:
import time

y_true_list = []
y_pred_list = []
t0 = time.perf_counter()
n = 0
for x_batch, y_batch in test_ds_s11:
    preds = model_s11.predict_on_batch(x_batch)
    y_pred_list.extend(np.argmax(preds, axis=1))
    y_true_list.extend(np.argmax(y_batch.numpy(), axis=1))
    n += len(y_batch)

infer_ms = (time.perf_counter() - t0) * 1000 / max(n, 1)
y_true_s11 = np.array(y_true_list)
y_pred_s11 = np.array(y_pred_list)
metrics_s11 = compute_metrics(y_true_s11, y_pred_s11)
save_scenario_metrics(
    11, enhancement, True, "cnn", "MobileNetV2",
    metrics_s11, infer_ms, len(y_true_s11), paths["metrics"],
)
plot_confusion_matrix(y_true_s11, y_pred_s11, title="Scenario 11 CNN Segmented",
                      save_path=paths["figures_confusion"] / "scenario_11.png")
metrics_s11


## Skenario 12: Pelatihan Model CNN Unsegmented (Citra Asli) + Raw

### Stage 1 — Base frozen (20 epoch)

In [ ]:
model_s12 = build_mobilenetv2(num_classes=2)
model_s12 = compile_mobilenet(model_s12, learning_rate=1e-4)
cb_s12 = get_mobilenet_callbacks(str(paths["models"] / "mobilenetv2_s12_stage1.h5"))

history1_s12 = model_s12.fit(
    train_ds_s12, validation_data=val_ds_s12, epochs=20,
    class_weight=class_weight, callbacks=cb_s12,
)


### Stage 2 — Fine-tune 20 lapisan terakhir (50 epoch)

In [ ]:
model_s12 = unfreeze_last_layers(model_s12, n=20)
model_s12 = compile_mobilenet(model_s12, learning_rate=1e-5)
cb2_s12 = get_mobilenet_callbacks(str(paths["models"] / "mobilenetv2_s12_stage2.h5"))

history2_s12 = model_s12.fit(
    train_ds_s12, validation_data=val_ds_s12, epochs=50,
    class_weight=class_weight, callbacks=cb2_s12,
)


### Evaluasi Skenario 12

In [ ]:
import time

y_true_list = []
y_pred_list = []
t0 = time.perf_counter()
n = 0
for x_batch, y_batch in test_ds_s12:
    preds = model_s12.predict_on_batch(x_batch)
    y_pred_list.extend(np.argmax(preds, axis=1))
    y_true_list.extend(np.argmax(y_batch.numpy(), axis=1))
    n += len(y_batch)

infer_ms = (time.perf_counter() - t0) * 1000 / max(n, 1)
y_true_s12 = np.array(y_true_list)
y_pred_s12 = np.array(y_pred_list)
metrics_s12 = compute_metrics(y_true_s12, y_pred_s12)
save_scenario_metrics(
    12, "none", False, "cnn", "MobileNetV2",
    metrics_s12, infer_ms, len(y_true_s12), paths["metrics"],
)
plot_confusion_matrix(y_true_s12, y_pred_s12, title="Scenario 12 CNN Unsegmented",
                      save_path=paths["figures_confusion"] / "scenario_12.png")
metrics_s12


## Grad-CAM (Skenario 11 — Segmented)

In [ ]:
import matplotlib.pyplot as plt

gradcam_dir = paths["figures_gradcam"]
gradcam_dir.mkdir(parents=True, exist_ok=True)
representative = ["Apple", "Banana", "Tomato"]

# Build the Grad-CAM sub-model ONCE and reuse it across all images
# (constructing a new tf.keras.Model per image re-allocates the graph).
grad_model = build_gradcam_model(model_s11)

for commodity in representative:
    for label in ["fresh", "rotten"]:
        subset = test_df[
            (test_df["commodity"].str.contains(commodity, case=False, na=False)) &
            (test_df["label"] == label)
        ]
        if subset.empty:
            subset = test_df[test_df["label"] == label].head(3)
        for _, row in subset.head(3).iterrows():
            out = process_image(path=row["filepath"], enhancement=enhancement, do_segment=True)
            if out["img"] is None:
                continue
            x = image_to_cnn_input(out["img"])
            heatmap = make_gradcam_heatmap(grad_model, x)
            fname = Path(row["filepath"]).stem
            save = gradcam_dir / f"{commodity}_{label}_{fname}.png"
            plot_gradcam(out["img"], heatmap, save_path=save)
            plt.close("all")


## McNemar Significance Tests

In [ ]:
import joblib

s6_path = paths["models"] / "svm_scenario_06.pkl"
if s6_path.exists():
    from src.experiments import extract_split_matrix
    s6_model = joblib.load(s6_path)
    X_test, y_test_s6, _ = extract_split_matrix(
        test_df, read_best_enhancement(paths["metrics"]), True, "all", paths["data_processed"]
    )
    y_pred_s6 = s6_model.predict(X_test)

    # 1. S11 vs S6
    stat11_s6, pval11_s6, conclusion11_s6 = mcnemar_test(y_true_s11, y_pred_s11, y_pred_s6)
    append_significance_test("S11 vs S6", "S11", "S6", stat11_s6, pval11_s6, conclusion11_s6, paths["metrics"])
    print('S11 vs S6:', stat11_s6, pval11_s6, conclusion11_s6)

    # 2. S12 vs S6
    stat12_s6, pval12_s6, conclusion12_s6 = mcnemar_test(y_true_s12, y_pred_s12, y_pred_s6)
    append_significance_test("S12 vs S6", "S12", "S6", stat12_s6, pval12_s6, conclusion12_s6, paths["metrics"])
    print('S12 vs S6:', stat12_s6, pval12_s6, conclusion12_s6)

    # 3. S11 vs S12
    stat11_12, pval11_12, conclusion11_12 = mcnemar_test(y_true_s11, y_pred_s11, y_pred_s12)
    append_significance_test("S11 vs S12", "S11", "S12", stat11_12, pval11_12, conclusion11_12, paths["metrics"])
    print('S11 vs S12:', stat11_12, pval11_12, conclusion11_12)
else:
    print("Jalankan notebook 02 terlebih dahulu untuk model S6.")
